# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

## 1. Question

*The research question and the decision it supports.*

This project asks whether historical search-performance signals can identify content pages at higher risk of impression decline.

The decision it supports is which pages should be prioritized for content review. The output is a ranked list that helps editors focus their attention on higher-priority pages instead of reviewing every page manually.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The project uses the FlyRank ML Internship dataset from the available warehouse release.

The main data source is `fact_content_daily_performance`, with query-level signals from `fact_content_query_90d` used to create model features.

The analysis uses historical observations to create the model inputs and a later observation window to define the `is_declining` outcome.

Client and content identifiers are anonymized and are used only for grouping and identifying rows, not as model features.

Fields that directly reveal the outcome, such as `trend_direction` and `trend_pct`, are excluded because they would introduce target leakage. Future performance information is also excluded from the model features.

In [24]:
df = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE gsc_data_available IS TRUE
      AND month IN ('2026-02', '2026-03')
    GROUP BY client_hash_id, content_hash_id, month
),

momentum AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE
            WHEN month = '2026-02' THEN impressions ELSE 0
        END) AS prev_month_impressions,

        SUM(CASE
            WHEN month = '2026-03' THEN impressions ELSE 0
        END) AS current_month_impressions

    FROM monthly
    GROUP BY client_hash_id, content_hash_id
),

query_signals AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share,
        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS kept_impressions
    FROM read_parquet(
        '{REL}/fact_content_query_90d.parquet'
    )
    GROUP BY content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.prev_month_impressions,
    m.current_month_impressions,

    q.visible_queries,
    q.rare_share,
    q.anon_share,

    q.top_query_impressions /
        NULLIF(q.kept_impressions, 0) AS top_query_share,

    CASE
        WHEN m.current_month_impressions
             < 0.8 * m.prev_month_impressions
        THEN 1
        ELSE 0
    END AS target

FROM momentum m

LEFT JOIN query_signals q
    ON m.content_hash_id = q.content_hash_id

WHERE m.prev_month_impressions > 0
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The model used is a Random Forest classifier. It was selected because it can capture non-linear relationships between search-performance signals and provides feature-importance values that can be inspected.

The model features are:

- prev_month_impressions
- visible_queries
- rare_share
- anon_share
- top_query_share

The target is `is_declining`, defined as whether current impressions are less than 80% of the previous comparison period.

The Week-4 baseline uses a simple rule that prioritizes high-visibility pages using impression and average-position thresholds.

The model is evaluated using a client-grouped train/test split. Clients in the test set are not present in the training set, providing a test of generalization to unseen clients.

Fields such as `trend_direction`, `trend_pct`, and future outcome information are excluded from the model features to reduce leakage.

In [25]:
feature_cols = [
    "prev_month_impressions",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]


model_data = df.dropna(subset=feature_cols).copy()

X = model_data[feature_cols]
y = model_data["target"]
groups = model_data["client_hash_id"]

print("Rows:", len(model_data))
print("Features:", feature_cols)
print("\nTarget distribution:")
print(y.value_counts(normalize=True))

Rows: 85475
Features: ['prev_month_impressions', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

Target distribution:
target
0    0.857397
1    0.142603
Name: proportion, dtype: float64


In [26]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))


Train rows: 54384
Test rows: 31091


In [27]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained successfully.")

model_pred = model.predict(X_test)

Model trained successfully.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The Random Forest is compared with the Week-4 baseline using the same test set and the same metrics. The comparison shows whether the learned model provides better predictive performance than the simple rule. The results are treated as decision-support rather than evidence of causal effects.

In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

model_pred = model.predict(X_test)

test_rows = model_data.iloc[test_idx][
    ["client_hash_id", "content_hash_id"]
].copy()

baseline_data = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_sum_position) /
        NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

test_rows = test_rows.merge(
    baseline_data,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

baseline_pred = (
    (
        (test_rows["gsc_impressions"] >= 1000) &
        (test_rows["gsc_avg_position"].between(4, 15))
    )
    .astype(int)
)

results = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, model_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, model_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, model_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, model_pred, zero_division=0)
    ]
})

results

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Method,Accuracy,Precision,Recall,F1
0,Week-4 Baseline,0.667524,0.096389,0.200778,0.130248
1,Random Forest,0.870155,0.366569,0.064851,0.110205


## 5. Limitations

*What this work cannot claim.*

This model should be treated as decision support, not as a definitive prediction of content performance.

The available data does not explain why a page gains or loses impressions. Factors such as changes in search intent, seasonality, content quality, and external events may affect performance but are not fully represented in the model.

The model can also produce false positives and false negatives, so a ranked recommendation should be reviewed by an editor before taking action.

The results describe observed patterns in the available dataset and do not establish causal relationships or explain Google's ranking algorithm.

## 6. Ranked recommendations

*The action playbook output the paper's recommendations section.*

The model output is used to create a ranked review queue.

Pages with higher estimated decline risk are placed higher in the queue. The recommended action is to review these pages for possible content updates rather than automatically changing or removing them.

The ranking is intended to help editors prioritize limited review time. A high-ranked page is a candidate for investigation, not proof that the page needs a specific change.

In [29]:

recommendations = model_data.iloc[test_idx][
    ["client_hash_id", "content_hash_id"]
].copy()

recommendations["decline_probability"] = model.predict_proba(X_test)[:, 1]

recommendations["rank"] = (
    recommendations["decline_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

recommendations["action"] = np.where(
    recommendations["decline_probability"] >= 0.70,
    "Review for Content Refresh",
    np.where(
        recommendations["decline_probability"] >= 0.50,
        "Monitor",
        "Low Priority"
    )
)

recommendations = recommendations.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

recommendations.head(20)

,client_hash_id,content_hash_id,decline_probability,rank,action
0,client_73cda7b4e4f265ea,content_12f8f5af4574bead,0.895,1,Review for Content Refresh
1,client_157ffe4d4a595515,content_ce9d662044b50053,0.895,2,Review for Content Refresh
2,client_73cda7b4e4f265ea,content_1c1b6df375d83a08,0.885,3,Review for Content Refresh
3,client_fef1a8f436438636,content_376b21273b88185a,0.880,4,Review for Content Refresh
4,client_73cda7b4e4f265ea,content_159b7b4dac8865ca,0.875,5,Review for Content Refresh
5,client_73cda7b4e4f265ea,content_99c7ddd9359238ba,0.865,6,Review for Content Refresh
6,client_fef1a8f436438636,content_b79338784b4a9f49,0.845,7,Review for Content Refresh
7,client_73cda7b4e4f265ea,content_8180b7c0b5da3da2,0.840,8,Review for Content Refresh
8,client_fef1a8f436438636,content_bc5db2ce76291460,0.830,9,Review for Content Refresh
9,client_73cda7b4e4f265ea,content_7832e2403deb109e,0.810,11,Review for Content Refresh


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper will include the following artifacts:

1. Model-versus-baseline performance table.
2. Feature-importance table showing which signals the model relied on most.
3. Ranked recommendation table showing the highest-priority pages for review.
4. Error analysis summarizing false positives and false negatives.

In [30]:
# Model vs baseline results table

results

,Method,Accuracy,Precision,Recall,F1
0,Week-4 Baseline,0.667524,0.096389,0.200778,0.130248
1,Random Forest,0.870155,0.366569,0.064851,0.110205


In [31]:
# Feature importance table

importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

importance

,Feature,Importance
0,prev_month_impressions,0.259180
3,anon_share,0.241762
2,rare_share,0.222968
4,top_query_share,0.165202
1,visible_queries,0.110887


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
